In [ ]:
!pip install pandas numpy tqdm pyarabic arabic-reshaper requests beautifulsoup4 openpyxl
import pandas as pd
import numpy as np
import json
import re
import os
import pickle
from tqdm import tqdm
from pyarabic.araby import strip_tashkeel, strip_tatweel
from google.colab import drive
import time
from datetime import datetime

# توصيل Google Drive
drive.mount('/content/drive')

# المسارات الأساسية
EXCEL_FOLDER = "/content/drive/MyDrive/Web_Scraping_Data_fatwa"
DRIVE_BASE = "/content/drive/MyDrive/fatwa_data_100k"
LOCAL_BASE = "/content/ultra_fast_fatwa_backup"
SAVE_PATH = "/content/drive/MyDrive/cleaned_fatwa_dataset"
os.makedirs(SAVE_PATH, exist_ok=True)

class AdvancedArabicTextCleaner:
    """فئة متقدمة لتنظيف النص العربي"""

    def __init__(self):
        self.setup_cleaner()

    def setup_cleaner(self):
        """إعداد قواعد التنظيف"""
        self.arabic_pattern = re.compile(r'[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFFa-zA-Z\s.,،؛:؟!ـ]')
        self.extra_spaces = re.compile(r'\s+')
        self.url_pattern = re.compile(r'http\S+|www\S+')
        self.html_pattern = re.compile(r'<.*?>')
        self.numbers_pattern = re.compile(r'\d+')  # نمط لإزالة الأرقام

        # قائمة الكلمات غير المرغوب فيها
        self.unwanted_phrases = [
            'أكمل القراءة', 'انظر أيضًا', 'المزيد', 'إقرأ المزيد', 'تابع القراءة',
            'شاهد أيضاً', 'مواضيع ذات صلة', 'اقرأ أيضا', 'المصدر', 'رابط الموضوع',
            'شارك هذه الفتوى', 'طباعة', 'تحميل', 'PDF', 'فيس بوك', 'تويتر'
        ]

    def clean_text_advanced(self, text, remove_numbers=True):
        """تنظيف متقدم للنص العربي مع إزالة الأرقام"""
        if pd.isna(text) or not isinstance(text, str):
            return ""

        # التنظيف الأساسي
        text = str(text).strip()

        # إزالة HTML tags
        text = self.html_pattern.sub(' ', text)

        # إزالة الروابط
        text = self.url_pattern.sub(' ', text)

        # إزالة التشكيل والتطويل
        text = strip_tashkeel(text)
        text = strip_tatweel(text)

        # إزالة الأرقام إذا طلب
        if remove_numbers:
            text = self.numbers_pattern.sub(' ', text)

        # إزالة الكلمات غير المرغوب فيها
        for phrase in self.unwanted_phrases:
            text = re.sub(re.escape(phrase), ' ', text, flags=re.IGNORECASE)

        # إزالة الرموز غير العربية (مع السماح بالحروف الإنجليزية)
        text = self.arabic_pattern.sub(' ', text)

        # تنظيف المسافات
        text = self.extra_spaces.sub(' ', text)

        return text.strip()

    def is_valid_content(self, text, min_length=20):
        """التحقق من صحة المحتوى"""
        if not text or len(text) < min_length:
            return False

        # التحقق من نسبة الحروف العربية
        arabic_chars = sum(1 for char in text if '\u0600' <= char <= '\u06FF')
        if arabic_chars < len(text) * 0.5:  # يجب أن يكون 50% عربي على الأقل
            return False

        return True

class DataPreprocessor:
    """فئة معالجة البيانات الرئيسية"""

    def __init__(self):
        self.cleaner = AdvancedArabicTextCleaner()

    def extract_qa_from_text(self, text):
        """استخراج السؤال والجواب من النص"""
        if not text:
            return None, None

        # تنظيف النص أولاً مع إزالة الأرقام من الإجابة فقط
        cleaned_text = self.cleaner.clean_text_advanced(text, remove_numbers=False)

        # أنماط للبحث عن السؤال والجواب
        patterns = [
            (r'(سؤال[:\s]*)([^؟]*[؟])', r'(جواب[:\s]*)(.*)'),
            (r'(السؤال[:\s]*)([^؟]*[؟])', r'(الجواب[:\s]*)(.*)'),
            (r'([^؟]*[؟])', r'(.*)'),  # نمط عام: أي شيء ينتهي بعلامة استفهام
        ]

        for q_pattern, a_pattern in patterns:
            question_match = re.search(q_pattern, cleaned_text)
            answer_match = re.search(a_pattern, cleaned_text)

            if question_match and answer_match:
                question = question_match.group(2 if question_match.lastindex >= 2 else 1).strip()
                answer = answer_match.group(2 if answer_match.lastindex >= 2 else 1).strip()

                # تنظيف الإجابة من العبارات غير المرغوب فيها والأرقام
                answer = self.clean_answer(answer)

                if question and answer and len(question) > 10 and len(answer) > 20:
                    return question, answer

        # إذا لم نجد نمط محدد، نقسم النص إلى جزأين
        sentences = re.split(r'[.!؟]', cleaned_text)
        if len(sentences) >= 2:
            question = sentences[0].strip() + '؟'
            answer = '. '.join(sentences[1:]).strip()
            answer = self.clean_answer(answer)

            if self.cleaner.is_valid_content(question) and self.cleaner.is_valid_content(answer):
                return question, answer

        return None, None

    def clean_answer(self, answer):
        """تنظيف الإجابة من العبارات الزائدة والأرقام"""
        # إزالة العبارات التي تبدأ الإجابة
        start_phrases = [
            'الجواب:', 'جواب:', 'الإجابة:', 'إجابة:', 'الحمد لله', 'بسم الله',
            'وعليكم السلام', 'السلام عليكم'
        ]

        for phrase in start_phrases:
            if answer.startswith(phrase):
                answer = answer[len(phrase):].strip()
                break

        # إزالة جميع الأرقام من الإجابة
        answer = re.sub(r'\d+', '', answer)

        # إزالة التواريخ الشائعة
        date_patterns = [
            r'\d{1,2}/\d{1,2}/\d{4}',
            r'\d{1,2}-\d{1,2}-\d{4}',
            r'\d{4}-\d{1,2}-\d{1,2}',
            r'في سنة \d+',
            r'عام \d+',
            r'سنة \d+'
        ]

        for pattern in date_patterns:
            answer = re.sub(pattern, '', answer)

        return answer.strip()

    def process_excel_row(self, row, source_file):
        """معالجة صف من بيانات Excel لاستخراج السؤال والجواب"""
        # البحث عن أعمدة السؤال والجواب المحتملة
        question_columns = ['question', 'سؤال', 'استفسار', 'query', 'title', 'عنوان']
        answer_columns = ['answer', 'جواب', 'إجابة', 'response', 'content', 'نص', 'محتوى']

        question_text = ""
        answer_text = ""

        # البحث عن السؤال في الأعمدة المخصصة
        for col in question_columns:
            if col in row and pd.notna(row[col]):
                question_text = str(row[col])
                break

        # البحث عن الجواب في الأعمدة المخصصة
        for col in answer_columns:
            if col in row and pd.notna(row[col]):
                answer_text = str(row[col])
                break

        # إذا لم نجد أعمدة منفصلة، نبحث في عمود المحتوى
        if not question_text or not answer_text:
            content_columns = ['content', 'text', 'نص', 'محتوى', 'question_answer', 'سؤال_جواب']
            for col in content_columns:
                if col in row and pd.notna(row[col]):
                    content = str(row[col])
                    q, a = self.extract_qa_from_text(content)
                    if q and a:
                        question_text, answer_text = q, a
                        break
            else:
                # إذا لم نستطع استخراج من المحتوى، نستخدم أول عمود كسؤال والباقي كجواب
                all_text = " ".join([str(row[col]) for col in row.index if pd.notna(row[col]) and isinstance(row[col], str)])
                q, a = self.extract_qa_from_text(all_text)
                if q and a:
                    question_text, answer_text = q, a

        # تنظيف النصوص - السؤال بدون إزالة أرقام، الإجابة مع إزالة أرقام
        cleaned_question = self.cleaner.clean_text_advanced(question_text, remove_numbers=False)
        cleaned_answer = self.cleaner.clean_text_advanced(answer_text, remove_numbers=True)

        # تنظيف إضافي للإجابة من الأرقام
        cleaned_answer = self.clean_answer(cleaned_answer)

        # التأكد من أن السؤال ينتهي بعلامة استفهام
        if cleaned_question and not cleaned_question.endswith('؟'):
            cleaned_question += '؟'

        # التحقق من صحة البيانات
        if (self.cleaner.is_valid_content(cleaned_question) and
            self.cleaner.is_valid_content(cleaned_answer) and
            len(cleaned_question) > 10 and
            len(cleaned_answer) > 30):  # زيادة الحد الأدنى للإجابة

            return {
                'question': cleaned_question,
                'answer': cleaned_answer  # الإجابة كاملة بدون اقتطاع وبدون أرقام
            }

        return None

    def load_excel_files(self, folder_path):
        """تحميل ملفات Excel من المجلد"""
        print("📂 جاري تحميل ملفات Excel...")
        qa_pairs = []

        if not os.path.exists(folder_path):
            print(f"❌ المجلد غير موجود: {folder_path}")
            return qa_pairs

        excel_files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls'))]
        print(f"✅ تم العثور على {len(excel_files)} ملف Excel")

        for file in tqdm(excel_files, desc="تحميل ملفات Excel"):
            try:
                file_path = os.path.join(folder_path, file)
                df = pd.read_excel(file_path)

                # معالجة كل صف
                for _, row in df.iterrows():
                    qa_pair = self.process_excel_row(row, file)
                    if qa_pair:
                        qa_pairs.append(qa_pair)

            except Exception as e:
                print(f"❌ خطأ في تحميل {file}: {e}")

        print(f"📊 تم استخراج {len(qa_pairs)} زوج سؤال/جواب من ملفات Excel")
        return qa_pairs

    def process_pickle_item(self, item):
        """معالجة عنصر من بيانات Pickle لاستخراج السؤال والجواب"""
        if not isinstance(item, dict):
            return None

        content = item.get('content', '')
        title = item.get('title', '')

        # تنظيف البيانات - السؤال بدون إزالة أرقام، الإجابة مع إزالة أرقام
        cleaned_content = self.cleaner.clean_text_advanced(content, remove_numbers=False)
        cleaned_title = self.cleaner.clean_text_advanced(title, remove_numbers=False)

        # محاولة استخراج السؤال والجواب من المحتوى
        question, answer = self.extract_qa_from_text(cleaned_content)

        # إذا لم نستطع استخراج من المحتوى، نستخدم العنوان كسؤال والمحتوى كجواب
        if not question and cleaned_title:
            question = cleaned_title
            if not question.endswith('؟'):
                question += '؟'
            answer = cleaned_content

        # إذا لم يكن هناك إجابة كافية، نتخطى هذا العنصر
        if not answer or len(answer) < 30:
            return None

        # تنظيف الإجابة من الأرقام
        answer = self.clean_answer(answer)

        # التحقق من صحة البيانات
        if (question and answer and
            self.cleaner.is_valid_content(question) and
            self.cleaner.is_valid_content(answer) and
            len(question) > 10 and
            len(answer) > 30):  # زيادة الحد الأدنى للإجابة

            return {
                'question': question,
                'answer': answer  # الإجابة كاملة بدون اقتطاع وبدون أرقام
            }

        return None

    def load_pickle_batches(self, base_paths):
        """تحميل بيانات الـ Pickle batches"""
        print("📂 جاري تحميل ملفات Pickle...")
        qa_pairs = []

        for base_path in base_paths:
            batch_dir = os.path.join(base_path, "batches")
            if not os.path.exists(batch_dir):
                print(f"⚠️ المجلد غير موجود: {batch_dir}")
                continue

            batch_files = sorted([f for f in os.listdir(batch_dir) if f.endswith('.pkl')])
            print(f"✅ تم العثور على {len(batch_files)} ملف في {batch_dir}")

            for batch_file in tqdm(batch_files, desc=f"معالجة {os.path.basename(base_path)}"):
                try:
                    file_path = os.path.join(batch_dir, batch_file)
                    with open(file_path, 'rb') as f:
                        batch_data = pickle.load(f)

                    for item in batch_data:
                        qa_pair = self.process_pickle_item(item)
                        if qa_pair:
                            qa_pairs.append(qa_pair)

                except Exception as e:
                    print(f"❌ خطأ في تحميل {batch_file}: {e}")

        print(f"📊 تم استخراج {len(qa_pairs)} زوج سؤال/جواب من ملفات Pickle")
        return qa_pairs

    def remove_duplicate_qa(self, qa_pairs):
        """إزالة أزواج السؤال/الجواب المكررة"""
        print("🔄 جاري إزالة البيانات المكررة...")

        seen_pairs = set()
        unique_pairs = []

        for pair in qa_pairs:
            # إنشاء بصمة فريدة للسؤال والجواب
            question_hash = hash(pair['question'][:200])  # زيادة طول السؤال للمقارنة
            answer_hash = hash(pair['answer'][:500])     # زيادة طول الجواب للمقارنة
            pair_hash = question_hash + answer_hash

            if pair_hash not in seen_pairs:
                seen_pairs.add(pair_hash)
                unique_pairs.append(pair)

        print(f"✅ تم إزالة التكرارات: {len(qa_pairs)} → {len(unique_pairs)} زوج")
        return unique_pairs

    def validate_qa_quality(self, qa_pairs):
        """التحقق من جودة أزواج السؤال/الجواب"""
        print("🔍 جاري التحقق من جودة البيانات...")

        valid_pairs = []
        quality_stats = {
            'total_processed': len(qa_pairs),
            'removed_short_question': 0,
            'removed_short_answer': 0,
            'removed_low_arabic': 0,
            'removed_no_question_mark': 0,
            'removed_has_numbers': 0
        }

        for pair in qa_pairs:
            question = pair['question']
            answer = pair['answer']

            # التحقق من طول السؤال
            if len(question) < 10:
                quality_stats['removed_short_question'] += 1
                continue

            # التحقق من طول الجواب
            if len(answer) < 30:  # زيادة الحد الأدنى للإجابة
                quality_stats['removed_short_answer'] += 1
                continue

            # التحقق من أن السؤال ينتهي بعلامة استفهام
            if not question.endswith('؟'):
                quality_stats['removed_no_question_mark'] += 1
                continue

            # التحقق من وجود أرقام في الإجابة
            if re.search(r'\d', answer):
                quality_stats['removed_has_numbers'] += 1
                continue

            # التحقق من نسبة الحروف العربية في السؤال
            arabic_chars_q = sum(1 for char in question if '\u0600' <= char <= '\u06FF')
            arabic_ratio_q = arabic_chars_q / len(question) if question else 0

            # التحقق من نسبة الحروف العربية في الجواب
            arabic_chars_a = sum(1 for char in answer if '\u0600' <= char <= '\u06FF')
            arabic_ratio_a = arabic_chars_a / len(answer) if answer else 0

            if arabic_ratio_q < 0.5 or arabic_ratio_a < 0.5:
                quality_stats['removed_low_arabic'] += 1
                continue

            valid_pairs.append(pair)

        quality_stats['final_count'] = len(valid_pairs)

        print(f"📊 إحصائيات الجودة:")
        print(f"   • المعالجة: {quality_stats['total_processed']}")
        print(f"   • المحذوفة (سؤال قصير): {quality_stats['removed_short_question']}")
        print(f"   • المحذوفة (جواب قصير): {quality_stats['removed_short_answer']}")
        print(f"   • المحذوفة (قليلة العربية): {quality_stats['removed_low_arabic']}")
        print(f"   • المحذوفة (بدون علامة استفهام): {quality_stats['removed_no_question_mark']}")
        print(f"   • المحذوفة (تحتوي أرقام): {quality_stats['removed_has_numbers']}")
        print(f"   • النهائية: {quality_stats['final_count']}")

        return valid_pairs, quality_stats

def main():
    """الدالة الرئيسية"""
    print("🚀 بدء عملية تنظيف ودمج البيانات...")
    start_time = time.time()

    # إنشاء المعالج
    processor = DataPreprocessor()

    # 1. تحميل بيانات Excel
    excel_qa_pairs = processor.load_excel_files(EXCEL_FOLDER)

    # 2. تحميل بيانات Pickle
    pickle_qa_pairs = processor.load_pickle_batches([DRIVE_BASE, LOCAL_BASE])

    # 3. دمج جميع البيانات
    all_qa_pairs = excel_qa_pairs + pickle_qa_pairs
    print(f"📊 إجمالي أزواج السؤال/الجواب المجمعة: {len(all_qa_pairs)}")

    if not all_qa_pairs:
        print("❌ لم يتم العثور على أي بيانات للمعالجة!")
        return

    # 4. إزالة التكرارات
    unique_qa_pairs = processor.remove_duplicate_qa(all_qa_pairs)

    # 5. التحقق من الجودة
    final_qa_pairs, quality_stats = processor.validate_qa_quality(unique_qa_pairs)

    if not final_qa_pairs:
        print("❌ لم تتبق أي بيانات صالحة بعد التنظيف!")
        return

    # 6. حفظ البيانات النهائية كـ JSON بسيط
    output_file = os.path.join(SAVE_PATH, "fatwa_qa_dataset.json")

    # حفظ البيانات مباشرة كقائمة من كائنات السؤال/الجواب
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_qa_pairs, f, ensure_ascii=False, indent=2)

    # 7. حفظ نسخة مختصرة للتحقق
    sample_file = os.path.join(SAVE_PATH, "sample_fatwa_qa.json")
    sample_data = final_qa_pairs[:min(50, len(final_qa_pairs))]

    with open(sample_file, 'w', encoding='utf-8') as f:
        json.dump(sample_data, f, ensure_ascii=False, indent=2)

    # 8. تقرير نهائي
    total_time = time.time() - start_time

    print("\n🎉 اكتملت عملية تنظيف ودمج البيانات!")
    print("=" * 50)
    print(f"📁 الملف النهائي: {output_file}")
    print(f"📁 ملف العينة: {sample_file}")
    print(f"📊 إجمالي أزواج السؤال/الجواب: {len(final_qa_pairs)}")
    print(f"⏱️ الوقت المستغرق: {total_time/60:.2f} دقيقة")

    # إحصائيات عن طول الإجابات
    answer_lengths = [len(pair['answer']) for pair in final_qa_pairs]
    if answer_lengths:
        avg_length = sum(answer_lengths) / len(answer_lengths)
        max_length = max(answer_lengths)
        min_length = min(answer_lengths)
        print(f"📏 متوسط طول الإجابات: {avg_length:.0f} حرف")
        print(f"📏 أطول إجابة: {max_length} حرف")
        print(f"📏 أقصر إجابة: {min_length} حرف")

    # التحقق من خلو الإجابات من الأرقام
    has_numbers_count = sum(1 for pair in final_qa_pairs if re.search(r'\d', pair['answer']))
    print(f"✅ الإجابات الخالية من الأرقام: {len(final_qa_pairs) - has_numbers_count}/{len(final_qa_pairs)}")

    if quality_stats['total_processed'] > 0:
        success_rate = (quality_stats['final_count'] / quality_stats['total_processed'] * 100)
        print(f"📈 نسبة النجاح: {success_rate:.1f}%")
    print("=" * 50)

    # عرض عينة من البيانات مع الإجابات الكاملة
    if final_qa_pairs:
        print("\n📄 عينة من البيانات النهائية (بإجابات كاملة وبدون أرقام):")
        for i, pair in enumerate(final_qa_pairs[:3]):
            print(f"{i+1}. السؤال: {pair['question']}")
            print(f"   الجواب: {pair['answer']}")
            print()

# التنفيذ
if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 بدء عملية تنظيف ودمج البيانات...
📂 جاري تحميل ملفات Excel...
✅ تم العثور على 6 ملف Excel


تحميل ملفات Excel: 100%|██████████| 6/6 [03:52<00:00, 38.79s/it]


📊 تم استخراج 92453 زوج سؤال/جواب من ملفات Excel
📂 جاري تحميل ملفات Pickle...
✅ تم العثور على 5000 ملف في /content/drive/MyDrive/fatwa_data_100k/batches


معالجة fatwa_data_100k: 100%|██████████| 5000/5000 [02:02<00:00, 40.90it/s]


⚠️ المجلد غير موجود: /content/ultra_fast_fatwa_backup/batches
📊 تم استخراج 28481 زوج سؤال/جواب من ملفات Pickle
📊 إجمالي أزواج السؤال/الجواب المجمعة: 120934
🔄 جاري إزالة البيانات المكررة...
✅ تم إزالة التكرارات: 120934 → 92999 زوج
🔍 جاري التحقق من جودة البيانات...
📊 إحصائيات الجودة:
   • المعالجة: 92999
   • المحذوفة (سؤال قصير): 0
   • المحذوفة (جواب قصير): 0
   • المحذوفة (قليلة العربية): 0
   • المحذوفة (بدون علامة استفهام): 0
   • المحذوفة (تحتوي أرقام): 0
   • النهائية: 92999

🎉 اكتملت عملية تنظيف ودمج البيانات!
📁 الملف النهائي: /content/drive/MyDrive/cleaned_fatwa_dataset/fatwa_qa_dataset.json
📁 ملف العينة: /content/drive/MyDrive/cleaned_fatwa_dataset/sample_fatwa_qa.json
📊 إجمالي أزواج السؤال/الجواب: 92999
⏱️ الوقت المستغرق: 6.26 دقيقة
📏 متوسط طول الإجابات: 987 حرف
📏 أطول إجابة: 31964 حرف
📏 أقصر إجابة: 31 حرف
✅ الإجابات الخالية من الأرقام: 92999/92999
📈 نسبة النجاح: 100.0%

📄 عينة من البيانات النهائية (بإجابات كاملة وبدون أرقام):
1. السؤال: لماذا يضرب الله الأمثال لنفسه في القرآن

تحميل البيانات المنظفة

In [ ]:
import json

with open("/content/drive/MyDrive/cleaned_fatwa_dataset/fatwa_qa_dataset.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

# معاينة البيانات
print(f"عدد أزواج السؤال/الجواب: {len(qa_data)}")
print(qa_data[:2])


عدد أزواج السؤال/الجواب: 92999
[{'question': 'لماذا يضرب الله الأمثال لنفسه في القرآن بخلقه؟', 'answer': 'أولا:يقول العلامة محمد الخضر حسين، رحمه الله: ضرب الله الأمثال في كتابه العزيز، دل على هذا الكتاب نفسه، فقال تعالى:وتلك الأمثال نضربها للناس لعلهم يتفكرون الحشر: ، وقال تعالى:وتلك الأمثال نضربها للناس وما يعقلها إلا العالمون العنكبوت: ، وقال تعالى:ولقد ضربنا للناس في هذا القرآن من كل مثل لعلهم يتذكرون الزمر: ... وللأمثال أثر بليغ في تلقي الدعوة بالقبول، لذلك أحرزت بين الأساليب التي يتحراها القرآن في هدايته منزلة سامية.... ويضرب المثل لتقرير حال الممثل في النفس؛ حيث يكون الممثل به أوضح من الممثل، أو يكون للنفس سابقة ألفة وائتناس به ، انتهى.انظر: موسوعة الأعمال الكاملة للإمام محمد الخضر حسين: ، بتصرف.ثانيا:قال تعالى: ضرب لكم مثلا من أنفسكم هل لكم من ما ملكت أيمانكم من شركاء في ما رزقناكم فأنتم فيه سواء تخافونهم كخيفتكم أنفسكم كذلك نفصل الآيات لقوم يعقلون الروم: .يقول ابن كثير، رحمه الله، في بيان وجه ضرب المثل في الآية الكريمة: هذا مثل ضربه الله تعالى للمشركين به، العابدين معه غيره، ا

اختيار Tokenizer

لأن البيانات عربية، أنسب خيار هو AraBERT أو أي نموذج BERT عربي.

نستخدم مكتبة transformers من HuggingFace:

In [ ]:
!pip install transformers

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenization

نقوم بتحويل السؤال والجواب إلى input_ids و attention_mask:


In [ ]:
questions = [item['question'] for item in qa_data]
answers = [item['answer'] for item in qa_data]

# Tokenization
tokenized_inputs = tokenizer(
    questions,
    padding=True,
    truncation=True,
    max_length=128,   # يمكنك تعديل الطول حسب البيانات
    return_tensors="pt"
)

tokenized_outputs = tokenizer(
    answers,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

print(tokenized_inputs.keys())  # input_ids, attention_mask
print(tokenized_outputs.keys())


KeysView({'input_ids': tensor([[   33,  2320,  8021,  ...,    31,    31,    31],
        [   33,  1081,  2627,  ...,    31,    31,    31],
        [   33,  1081,   965,  ...,    31,    31,    31],
        ...,
        [   33,   736,  1509,  ..., 30404,   199,    34],
        [   33,   414, 10093,  ...,   369,  1541,    34],
        [   33, 48924,   195,  ..., 52578,  1898,    34]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])})
KeysView({'input_ids': tensor([[   33, 17003,    59,  ...,  2777,  1006,    34],
        [   33, 17003,    59,  ...,  4269,   596,    34],


تجهيز Dataset للتدريب

يمكن استخدام PyTorch DataLoader:

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):
    def __init__(self, input_encodings, output_encodings):
        self.input_encodings = input_encodings
        self.output_encodings = output_encodings

    def __len__(self):
        return len(self.input_encodings['input_ids'])

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.input_encodings.items()}
        item['labels'] = self.output_encodings['input_ids'][idx]
        return item

dataset = QADataset(tokenized_inputs, tokenized_outputs)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


شرح الخطوات:

تثبيت المكتبات: transformers, datasets, torch, sentencepiece ضرورية لتشغيل T5 عربي.

تحميل البيانات النهائية: JSON من مرحلة التنظيف، استخراج السؤال والجواب.

تحميل Tokenizer: لتحويل النصوص العربية إلى tokens.

Tokenization: تحويل أسئلة وأجوبة إلى input_ids و attention_mask جاهزة للنموذج.

Dataset وDataLoader: تغذية النموذج بالبيانات بكفاءة.

تحميل نموذج T5 عربي: Encoder-Decoder لتوليد الإجابات.

Trainer: تحكم كامل في التدريب (epochs, batch size, logging, save steps).

بدء التدريب: النموذج يبدأ التعلم على البيانات.

حفظ النموذج النهائي: جاهز للاستخدام لتوليد إجابات